# The protocol and its primitives

**Scenario:** a music service runs four assistants over five internal systems. Catalogue, listening
history, rights, playlists, label notes. Every assistant needs every system, so somebody writes twenty
connectors and nobody owns them.

The Model Context Protocol replaces those with one agreement. Think of **a wall socket**. No appliance
ships with its own generator, because the shape of the plug was settled once.

Then the first host is built against it, and a failed tool call is read as a success. Two spellings of
one idea sit an underscore apart.

## Mechanics

Every message is JSON-RPC 2.0: one JSON object, on one line.

| Field | On | Meaning |
|---|---|---|
| `jsonrpc` | every message | Always the string `2.0` |
| `id` | request and reply | Matches a reply to its request. No `id` means a notification, and gets no reply |
| `method` | request | What you are asking for, such as `tools/list` |
| `params` | request | The arguments, as an object |
| `result` | reply | The answer, when it worked |
| `error` | reply | A code and a message, when the request itself was rejected |

A server offers three kinds of thing. The difference is who chooses to use them.

| Primitive | Listed by | Fetched by | Who picks it |
|---|---|---|---|
| Tool | `tools/list` | `tools/call` | The model, from the description and the written shape of the arguments |
| Resource | `resources/templates/list` | `resources/read` | The host or the person, by URI |
| Prompt | `prompts/list` | `prompts/get` | The person, usually from a menu |

One trap outweighs the rest of that table. On the wire a tool carries `inputSchema` and a call result
carries `isError`, both camel case. The Messages API you write the host against spells the same two
things `input_schema` and `is_error`.

## The picture

![One protocol replaces a connector for every pair](images/m-by-n.svg)

Left is what a team builds with no agreement. Right is what it builds with one.

## The cost

```
without a protocol : connectors = hosts x systems
with one           : connectors = hosts + systems
```

Four assistants over five systems is twenty pieces of glue to write and keep alive. The same estate
behind one protocol is nine.

## The failure

Here is a server with one of each primitive. It knows nothing about a model, and three decorators are
its whole surface.

In [1]:
import json
import pathlib

SERVER = pathlib.Path("servers/catalogue.py")

for line in SERVER.read_text().splitlines():
    if line.startswith(("@srv.", "def ")):
        print(line)

@srv.tool()
def similar_tracks(track_id: str, limit: int = 3) -> list[str]:
@srv.resource("catalogue://track/{track_id}")
def track_record(track_id: str) -> str:
@srv.prompt()
def explain_pick(track_id: str) -> str:


Now we speak to it by hand, because the protocol is small enough to type. This helper writes request
lines into the server and reads reply lines back.

In [2]:
import subprocess
import sys
import threading


def speak(script, requests, expect, timeout=20):
    """Send hand written JSON-RPC lines to a server, read the replies back."""
    proc = subprocess.Popen([sys.executable, script], text=True,
                            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE)
    watchdog = threading.Timer(timeout, proc.kill)
    watchdog.start()
    try:
        proc.stdin.write("".join(json.dumps(r) + "\n" for r in requests))
        proc.stdin.flush()
        return [proc.stdout.readline().rstrip("\n") for _ in range(expect)]
    finally:
        watchdog.cancel()
        proc.kill()
        proc.wait(timeout=5)

The watchdog matters. A server that never answers would hold this cell open forever otherwise.

Then the messages: a handshake, the notification that closes it, a request for the tool list, and
three calls. Two of the three are meant to fail.

In [3]:
def rpc(request_id, method, **params):
    """One JSON-RPC request line, exactly as it goes down the pipe."""
    return {"jsonrpc": "2.0", "id": request_id, "method": method, "params": params}


READY = {"jsonrpc": "2.0", "method": "notifications/initialized"}
HELLO = rpc(1, "initialize", protocolVersion="2025-06-18", capabilities={},
            clientInfo={"name": "by-hand", "version": "0"})
CALLS = [rpc(n, "tools/call", name="similar_tracks", arguments={"track_id": t})
         for n, t in enumerate(["T-1042", "T-9999", 42], start=3)]

wire = speak(SERVER, [HELLO, READY, rpc(2, "tools/list"), *CALLS], expect=5)
print(wire[0][:118])
print(wire[1][:118])

{"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2025-06-18","capabilities":{"experimental":{},"prompts":{"listCha
{"jsonrpc":"2.0","id":2,"result":{"tools":[{"name":"similar_tracks","description":"Track ids a listener of this one us


That is the real text on the pipe. Now the part a host gets wrong: it reads those replies with the
field names it already knows.

In [4]:
tool = json.loads(wire[1])["result"]["tools"][0]
results = [json.loads(line)["result"] for line in wire[2:]]

print("keys on the tool spec :", sorted(tool))
print("keys on a call result :", sorted(results[0]))

missed = [r for r in results if not r.get("is_error")]
print(f"\nhost read is_error, and saw {len(missed)} of {len(results)} calls as fine")
print(f"the wire says isError on {sum(r['isError'] for r in results)} of them")

assert "input_schema" in tool, "the host wants input_schema and the wire has no such key"

keys on the tool spec : ['description', 'inputSchema', 'name', 'outputSchema']
keys on a call result : ['content', 'isError', 'structuredContent']

host read is_error, and saw 3 of 3 calls as fine
the wire says isError on 2 of them


AssertionError: the host wants input_schema and the wire has no such key

## The diagnosis

Two calls failed and the host counted zero. The assertion is the loud half of the same bug.

**`dict.get` is why it was silent.** `results[0].get("is_error")` returns `None`, which is falsy, which
reads as no error. Every failed call then flows on as a result and the model is told it worked.

**Neither spelling is wrong.** The wire is camel case because JSON-RPC is. The library is snake case
because Python is. The host sits between them, so translating is its job.

**`isError` is not a JSON-RPC `error`.** The first means your function raised. The second means the
request never reached it. Fold them together and you cannot tell a broken tool from a broken server.

## The fix

One place translates, and it breaks loudly when the wire is not the shape it expected.

In [5]:
def to_host_tool(spec):
    """Wire shape to the shape a host tool definition needs. Missing keys raise."""
    return {"name": spec["name"],
            "description": spec.get("description", ""),
            "input_schema": spec["inputSchema"]}

`spec["inputSchema"]` is a direct read on purpose. A tool with no schema is not one a host should
offer, so this raises rather than passing an empty dict on.

Then the half that was silent.

In [6]:
def failed(result):
    """True when the server ran the tool and the tool did not work."""
    if "isError" not in result:
        raise KeyError("no isError on this result, the reply is not an MCP tool result")
    return bool(result["isError"])

Now count the same three calls with both readers. Nothing about the server changed.

In [7]:
before = [r for r in results if r.get("is_error")]
after = [r for r in results if failed(r)]

print(f"calls made          : {len(results)}")
print(f"failures seen before: {len(before)}")
print(f"failures seen after : {len(after)}")
print(f"first error text    : {after[0]['content'][0]['text'][:64]}")
print(f"\nadapted spec: {to_host_tool(tool)['input_schema']['required']}")

calls made          : 3
failures seen before: 0
failures seen after : 2
first error text    : Error executing tool similar_tracks: unknown track T-9999

adapted spec: ['track_id']


## The gate

The regression to stop is a second reader appearing elsewhere in the old spelling. This test needs no
server and no model.

In [8]:
def test_camel_case_is_the_only_reader():
    assert failed({"content": [], "isError": True}), "isError was ignored"
    assert not failed({"content": [], "isError": False}), "a good call was flagged"
    try:
        to_host_tool({"name": "x", "description": "y"})
    except KeyError:
        return
    raise AssertionError("a tool with no inputSchema was accepted")


test_camel_case_is_the_only_reader()
print("gate holds: a failed call cannot be read as a success")

gate holds: a failed call cannot be read as a success


Change `failed` to use `result.get("is_error")` and this test fails on its first line.

### Enterprise exploration

- Twenty connectors became nine. What is the cost of one going down now that four assistants share it?
- A server can rename a tool between two `tools/list` calls. What does your host cache, and for how
  long, before a stale schema fails calls at scale?
- Tools are picked by the model, prompts and resources by a person. Which of those needs an audit trail
  in a regulated setting, and who signs off the list?

### Key takeaways

- One protocol turns a connector per pair into an adapter per side.
- Tools are picked by the model, resources and prompts by a person. That decides where you gate them.
- The wire is camel case. Your host library is probably not.
- `dict.get` on a name that does not exist is the quietest bug here.